In [1]:
!pip install tldextract Levenshtein python-whois

# 하는 일: 링크를 쪼개고, 글자 차이를 세고, 도메인 정보를 조회하는 도구를 설치함
# 이유: 이 기능들이 파이썬 기본에는 없기 때문

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.0 MB/s eta 0:00:00


In [2]:
import tldextract

ext = tldextract.extract("http://naver-login.com/verify")
print(ext.subdomain, ext.domain, ext.suffix)

# 하는 일: 주소를 앞부분(subdomain) / 이름(domain) / 끝(suffix)으로 나눔
# 이유: 진짜 이름 부분만 골라야 유명 이름을 따라 했는지 비교할 수 있음

 naver-login com


In [3]:
import Levenshtein

print(Levenshtein.distance("naver-login", "naver"))
print(Levenshtein.distance("taiegram", "telegram"))

# 하는 일: 한 이름을 다른 이름으로 바꾸려면 글자를 몇 개 고쳐야 하는지 셈 (작을수록 비슷함)
# 이유: telegram → taiegram처럼 글자만 살짝 바꾼 가짜 주소를 잡기 위해

6
2


In [4]:
import whois

try:
    w = whois.whois("google.com")
    print(w.creation_date)
except Exception as e:
    print("조회 실패:", e)

# 하는 일: 인터넷 등록 장부에서 도메인이 언제 만들어졌는지 날짜를 가져옴
# 이유: 얼마 전에 급하게 만든 도메인일수록 피싱일 가능성이 높기 때문
# 참고: 날짜가 목록으로 나올 수 있음. 조회가 실패하는 것도 정상이라 try/except로 감쌈

[datetime.datetime(1997, 9, 15, 4, 0, tzinfo=tzoffset('UTC', 0)), datetime.datetime(1997, 9, 15, 7, 0, tzinfo=tzoffset('UTC', 0))]


In [5]:
import json
import datetime
import tldextract
import Levenshtein
import whois

# 진짜 이름 목록 (사례조사에서 나온 것부터 시작. 나중에 늘리기)
BRANDS = ["naver", "telegram", "kakao", "google"]

def analyze_url_sample(url, use_whois=True):
    reasons = []        # 점수가 나온 이유를 모아 두는 목록
    unavailable = []    # 확인하지 못한 항목을 모아 두는 목록

    ext = tldextract.extract(url)
    name = ext.domain.lower()
    # 하는 일: 주소에서 이름 부분만 뽑음 (셀 2에서 연습한 것)

    # 1. 도메인 유사 (최대 35점)
    for brand in BRANDS:
        if name != brand and (brand in name or Levenshtein.distance(name, brand) <= 2):
            reasons.append({
                "factor": "도메인 유사",
                "points": 35,
                "text": f"유명 이름('{brand}')과 비슷하지만 똑같지는 않은 도메인입니다."
            })
            break
    # 하는 일: 진짜 이름이 들어 있거나 글자가 1~2개만 다르면 점수를 줌 (셀 3에서 연습한 것)
    # 이유: 진짜 이름과 똑같으면 진짜 사이트이므로 점수를 주지 않음 (name != brand)

    # 2. 키워드 (최대 10점)
    if "verify" in url.lower() or "login" in url.lower():
        reasons.append({
            "factor": "키워드",
            "points": 10,
            "text": "주소에 verify, login처럼 인증을 유도하는 단어가 들어 있습니다."
        })
    # 하는 일: 주소에 인증을 유도하는 단어가 있으면 점수를 줌
    # 이유: 피싱 주소에는 로그인이나 인증을 하라는 단어가 자주 들어가기 때문

    # 3. 도메인 생성일 (최대 30점): 만든 지 30일 미만이면 점수
    if use_whois:
        try:
            if not ext.suffix:
                raise ValueError("도메인 형태가 아님")
            info = whois.whois(f"{ext.domain}.{ext.suffix}")
            created = info.creation_date
            if isinstance(created, list):      # 날짜가 목록으로 나오면 첫 번째 것을 씀
                created = created[0]
            if created is None:
                raise ValueError("생성일 정보 없음")
            if created.tzinfo is None:         # 시간대 정보가 없으면 UTC로 맞춤
                created = created.replace(tzinfo=datetime.timezone.utc)
            days = (datetime.datetime.now(datetime.timezone.utc) - created).days
            if days < 30:
                reasons.append({
                    "factor": "생성일",
                    "points": 30,
                    "text": f"도메인이 만들어진 지 {days}일밖에 안 됐습니다."
                })
        except Exception:
            unavailable.append("whois")
    else:
        unavailable.append("whois")
    # 하는 일: 도메인이 만들어진 지 얼마나 됐는지 조회함 (셀 4에서 연습한 것)
    # 이유: 조회에 실패하면 0점이 아니라 unavailable에 적어서 "확인 못 했다"는 걸 구분함

    # 4. 점수 합치기
    score = sum(r["points"] for r in reasons)
    # 하는 일: 이유별 점수를 모두 더함
    # 이유: README 약속 1번 "score는 reasons의 points를 더한 값"

    # 5. 팀 약속 모양으로 돌려주기
    return {
        "type": "url",
        "input": url,
        "score": score,
        "model_probability": None,   # 아직 모델이 없어서 비어 있음 (JSON에서는 null이 됨)
        "reasons": reasons,
        "unavailable": unavailable
    }
    # 이유: A가 만든 결과를 B(화면)와 C(장부)가 같은 칸 이름으로 받아 쓰게 하려고

# 실행해 보기 (출력된 링크는 누르지 않기)
result = analyze_url_sample("http://naver-login.com/verify")
print(json.dumps(result, ensure_ascii=False, indent=2))

ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out


{
  "type": "url",
  "input": "http://naver-login.com/verify",
  "score": 45,
  "model_probability": null,
  "reasons": [
    {
      "factor": "도메인 유사",
      "points": 35,
      "text": "유명 이름('naver')과 비슷하지만 똑같지는 않은 도메인입니다."
    },
    {
      "factor": "키워드",
      "points": 10,
      "text": "주소에 verify, login처럼 인증을 유도하는 단어가 들어 있습니다."
    }
  ],
  "unavailable": []
}


In [6]:
# 확인 1: naver-login.com의 생성일이 실제로 어떻게 나오는지 보기 (등록 정보만 조회하고 접속은 하지 않음)
try:
    info = whois.whois("naver-login.com")
    print("생성일:", info.creation_date)
except Exception as e:
    print("조회 실패:", e)

# 확인 2: 진짜 이름과 철자만 바꾼 이름이 잘 구분되는지 (whois는 건너뜀)
tests = ["http://google.com/", "http://taiegram.com/login"]
for u in tests:
    r = analyze_url_sample(u, use_whois=False)
    print(u, "→ 점수", r["score"], "| 이유", [x["factor"] for x in r["reasons"]], "| 확인 못 함", r["unavailable"])

# 하는 일: 위 추측이 맞는지 보고, 진짜 이름은 점수 0, 철자 바꾼 이름은 점수가 붙는지 확인함
# 이유: 점수 함수가 "진짜는 안전, 가짜는 위험"으로 나뉘는지 시험하기 위해

ERROR:whois.whois:Error trying to connect to socket: closing socket - timed out


생성일: 2026-03-17 00:41:38+00:00
http://google.com/ → 점수 0 | 이유 [] | 확인 못 함 ['whois']
http://taiegram.com/login → 점수 45 | 이유 ['도메인 유사', '키워드'] | 확인 못 함 ['whois']
